# Steps 6-7: Baseline & Hybrid Model Training

Train PD (Logistic Regression), LGD (Beta Regression), EAD (Linear Regression).
Compare baseline (structured only) vs hybrid (structured + NLP) PD models.
Compute Expected Loss: EL = PD x LGD x EAD.

In [ ]:
import sys
sys.path.insert(0, '/home/jovyan/work/src')

import pandas as pd
import numpy as np
from spark_utils import get_spark_session
from models import train_pd_model, train_lgd_model, train_ead_model, compute_expected_loss, get_model_coefficients

## 5.1 Load Feature Matrices

In [ ]:
spark = get_spark_session()

baseline_df = spark.read.parquet('/home/jovyan/work/data/parquet/features_baseline.parquet').toPandas()
hybrid_df = spark.read.parquet('/home/jovyan/work/data/parquet/features_hybrid.parquet').toPandas()

## 5.2 Prepare Feature Sets

In [ ]:
# Baseline: WoE-encoded structured features only
baseline_woe_cols = [c for c in baseline_df.columns if c.endswith('_woe')]
X_baseline = baseline_df[baseline_woe_cols].fillna(0)
y = baseline_df['default_flag']

# Hybrid: WoE + sentiment + TF-IDF + occupation cluster + missing flags
nlp_cols = [c for c in hybrid_df.columns if any(c.startswith(p) for p in ['tfidf_', 'desc_vader', 'desc_textblob', 'title_vader', 'title_textblob', 'desc_missing', 'title_missing', 'occupation_cluster'])]
hybrid_feature_cols = baseline_woe_cols + nlp_cols
X_hybrid = hybrid_df[hybrid_feature_cols].fillna(0)

print(f'Baseline features: {len(baseline_woe_cols)}')
print(f'Hybrid features: {len(hybrid_feature_cols)}')

## 5.3 Train Baseline PD Model

In [ ]:
model_baseline, X_train_b, X_test_b, y_train_b, y_test_b, y_pred_baseline = train_pd_model(X_baseline, y)
print(f'Baseline PD model trained. Test size: {len(y_test_b)}')

## 5.4 Train Hybrid PD Model

In [ ]:
model_hybrid, X_train_h, X_test_h, y_train_h, y_test_h, y_pred_hybrid = train_pd_model(X_hybrid, y)
print(f'Hybrid PD model trained. Test size: {len(y_test_h)}')

## 5.5 Train LGD Model (Beta Regression)

In [ ]:
# LGD model on defaulted loans only
default_mask = baseline_df['default_flag'] == 1
X_lgd = baseline_df.loc[default_mask, baseline_woe_cols].fillna(0)
y_lgd = baseline_df.loc[default_mask, 'lgd'].fillna(0.5)

lgd_model, lgd_summary = train_lgd_model(X_lgd, y_lgd)
print(lgd_summary)

## 5.6 Train EAD Model (Linear Regression)

In [ ]:
X_ead = baseline_df[baseline_woe_cols].fillna(0)
y_ead = baseline_df['ead'].fillna(0)

ead_model, ead_predictions = train_ead_model(X_ead, y_ead)
print(f'EAD R-squared: {ead_model.score(X_ead, y_ead):.4f}')

## 5.7 Compute Expected Loss

In [ ]:
# Use test set predictions for EL
# Note: LGD and EAD are shared between baseline and hybrid (only PD differs)
# For simplicity, use mean LGD and EAD for EL computation

mean_lgd = y_lgd.mean()
mean_ead = y_ead.mean()

el_baseline = compute_expected_loss(y_pred_baseline, mean_lgd, mean_ead)
el_hybrid = compute_expected_loss(y_pred_hybrid, mean_lgd, mean_ead)

print(f'Mean Expected Loss (Baseline): ${el_baseline.mean():,.2f}')
print(f'Mean Expected Loss (Hybrid): ${el_hybrid.mean():,.2f}')

## 5.8 Save Results

In [ ]:
# Save predictions for evaluation notebook
results = pd.DataFrame({
    'y_true': y_test_b,
    'pd_baseline': y_pred_baseline,
    'pd_hybrid': y_pred_hybrid,
    'el_baseline': el_baseline,
    'el_hybrid': el_hybrid,
})
results.to_csv('/home/jovyan/work/outputs/tables/model_predictions.csv', index=False)
print('Predictions saved.')